In [1]:
import pandas as pd
import json
import base64
import time
from pathlib import Path
from datetime import datetime
from pdf2image import convert_from_path
import mimetypes
from typing import Optional, List, Dict

from anthropic import Anthropic


# ==========================
# CONFIGURATION (COMPANY GATEWAY)
# ==========================
base_url = "https://anthropic.prod.ai-gateway.quantumblack.com/7a4f3d63-b5db-4d5b-8076-8f114d1f14f7"
access_token = "eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJhZXNKN2kxNGNidnVuTU40MTJrOU5yZ2ROeENhTlJudTNPbC1TU08ycFlJIn0.eyJleHAiOjE3NjUyNTY1MjksImlhdCI6MTc2NTI1NDcyOSwiYXV0aF90aW1lIjoxNzY1MjU0NzI4LCJqdGkiOiI4ZTU0ZDgwOC05MmU5LTQ3YTYtYWUyZi0zNGQzYjk5MWZmYmIiLCJpc3MiOiJodHRwczovL2F1dGgubWNraW5zZXkuaWQvYXV0aC9yZWFsbXMvciIsImF1ZCI6ImJjZDIzNzI4LTNkMjctNDQ3Yy1hMGE5LWVhY2FmMzkzYTZmNSIsInN1YiI6IjI0NDRiYzZjLTAwMzctNGIyZS1hYzI3LWZjNTlhNTkxNTM2NiIsInR5cCI6IklEIiwiYXpwIjoiYmNkMjM3MjgtM2QyNy00NDdjLWEwYTktZWFjYWYzOTNhNmY1Iiwic2Vzc2lvbl9zdGF0ZSI6IjRlOTFiMWVhLTY4M2UtNDI1My05Zjg3LTNlYzhlMzAyMmQ1MiIsImF0X2hhc2giOiJ0QzF4NWRuRnhMZWw1X3AzaHFfRlZBIiwibmFtZSI6IlVnYW5kaGFyIFZhZGRpIiwiZ2l2ZW5fbmFtZSI6IlVnYW5kaGFyIiwiZmFtaWx5X25hbWUiOiJWYWRkaSIsInByZWZlcnJlZF91c2VybmFtZSI6IjE1ZDhiYmNkMmMzNTNmYWUiLCJlbWFpbCI6IlVnYW5kaGFyX1ZhZGRpQG1ja2luc2V5LmNvbSIsImFjciI6IjEiLCJzaWQiOiI0ZTkxYjFlYS02ODNlLTQyNTMtOWY4Ny0zZWM4ZTMwMjJkNTIiLCJlbWFpbF92ZXJpZmllZCI6dHJ1ZSwiZm1ubyI6IjM0NzI3NiIsImdyb3VwcyI6WyI3YTRmM2Q2My1iNWRiLTRkNWItODA3Ni04ZjExNGQxZjE0ZjciLCJBbGwgRmlybSBVc2VycyJdfQ.df9PWvr-r_hg_tZzleq6OZ6a8cSWdCJ403yb2EPQ9dtbNVI8amb5wgkV5Vl29rDNc9VBdZfepi-K4mx5S44SADHVxEcnQnK8gfik-t4FA-ZfjlDxgNdcPQhPSEjCw7tBALCcYPKGcDYWBT8YECuPNExPhuvCX5IY5IQ7zjrGMZWzoq-u2XZcXzhp1HK2KEoEXBuFz3i3tsqN280syj2dpXofiAJHqDLeuKxWp2YMBJN6_mhtEmo5uBeff7OWEKe7F7oFaiVFFNLIl3Ae5SdFIT98ItHdj_MooHB1XK5flk1vXBKtl2EhplM93Ci7WWGZa8QGgUPoAoVMDpRqNM6BkQ"

client = Anthropic(
    base_url=base_url,
    api_key=access_token
)

# Using Claude Sonnet 4.5 for best balance of accuracy and speed
MODEL_NAME = "claude-sonnet-4-5-20250929"


# ==========================
# FOCUSED PROMPT - 6 CRITICAL PARAMETERS ONLY
# ==========================
def get_dimension_prompt():
    return """
    You are an expert Engineering Drawing Analyzer with exceptional spatial understanding 
    and visual recognition capabilities. Your task is to extract EXACTLY 6 parameters from 
    mechanical engineering drawings.

    COGNITIVE APPROACH - SPATIAL ANALYSIS:
    1. Scan the ENTIRE drawing to understand layout, orientation, and scale
    2. Identify the title block (usually bottom-right corner)
    3. Locate dimension lines and their corresponding values
    4. Find material callouts and annotations anywhere on the drawing
    5. Look for calculated properties (weight, surface area) in title block or notes
    6. Cross-validate extracted values with visual geometry

    ═══════════════════════════════════════════════════════════════════
    REQUIRED PARAMETERS (8 TOTAL):
    ═══════════════════════════════════════════════════════════════════

    TITLE BLOCK INFORMATION (ALWAYS CHECK FIRST):
    
    0A. DRAWING TITLE
        - Main title or name of the part/assembly
        - May be labeled: TITLE, PART NAME, DRAWING TITLE, DESCRIPTION
        - Usually in title block (bottom-right corner)
        - Examples: "MOUNTING BRACKET", "SHAFT ASSEMBLY", "SPACER RING"
        - Extract exact text as shown
    
    0B. DRAWING NUMBER
        - Unique identifier for the drawing
        - May be labeled: DWG NO., DRAWING NO., PART NO., P/N, DRG NO.
        - Usually in title block (bottom-right corner)
        - May include revision (e.g., "DRW-12345-A", "P/N: 987654 REV B")
        - Extract complete number including revision if shown

    DIMENSIONAL PARAMETERS:

    1. LENGTH
       - Overall length of the part (primary/longest dimension)
       - May be labeled: L, LENGTH, LG, OVERALL LENGTH
       - Look for dimension lines along the longest axis
       - Extract with unit (mm, cm, m, in, inch, inches, ft)

    2. INNER DIAMETER (ID)
       - Internal diameter of holes, bores, or hollow sections
       - May be labeled: ID, I.D., INNER DIA, BORE, Ø (with "inner" context)
       - Often shown in section views (SECTION A-A)
       - Extract with unit (mm, cm, m, in, inch, inches)

    3. OUTER DIAMETER (OD)
       - External diameter of cylindrical parts, shafts, rings
       - May be labeled: OD, O.D., OUTER DIA, DIA, Ø
       - Shown in front/side views
       - Extract with unit (mm, cm, m, in, inch, inches)

    4. MATERIAL
       - **CRITICAL**: Material can be specified in MULTIPLE ways:
       
       A) Direct Text Labels:
          - MATL, MATERIAL, MAT'L, MAT, MTRL
          - Examples: "ALUMINUM 6061-T6", "STEEL 1045", "SS 316"
       
       B) Annotation Symbols/Callouts:
          - Look for circled numbers (①, ②, ③) or letters (Ⓐ, Ⓑ) on the drawing
          - These reference a material table/legend usually at bottom or side
          - Example: Part shows "①" → Check table: "① = ALUMINUM 6061"
       
       C) Leader Lines with Material:
          - Arrow pointing to part with material text
          - Example: Arrow → "316 STAINLESS STEEL"
       
       D) Title Block Material Field:
          - Dedicated "MATERIAL:" field in title block
       
       E) Notes Section:
          - "NOTE: MATERIAL IS..." or "ALL PARTS: [material]"
       
       **SEARCH STRATEGY**:
       - First check title block for MATERIAL field
       - Then scan entire drawing for annotation symbols (circles, numbers)
       - If you find ① or similar, locate the corresponding table/legend
       - Check notes section at bottom/side
       - Look for leader lines pointing to the part
       - Extract FULL material specification including grade/temper

    5. WEIGHT (WT)
       - Part weight/mass
       - May be labeled: WT, WEIGHT, Wt., MASS, WGT
       - **UNIT REQUIREMENT**: Extract ONLY if in pounds (lb, lbs)
       - If shown in kg, g, oz, N → extract value BUT keep original unit
       - Common locations: title block, bottom-right, notes section
       - Extract with unit (lb, lbs, kg, g, oz)

    6. SURFACE AREA (S/A)
       - Total surface area of the part
       - May be labeled: S/A, SURF AREA, SURFACE AREA, SA, AREA
       - **UNIT REQUIREMENT**: Extract ONLY if in square inches (in², in^2, sq in)
       - If shown in cm², m², mm² → extract value BUT keep original unit
       - Common locations: title block, notes section
       - Extract with unit (in^2, in², cm², m², mm²)

    ═══════════════════════════════════════════════════════════════════
    EXTRACTION RULES:
    ═══════════════════════════════════════════════════════════════════

    UNIT HANDLING:
    - Extract PRIMARY unit shown (leftmost or topmost if dual units present)
    - If dimensions shown as "100mm [3.94in]" → extract 100mm
    - Ignore reference dimensions in parentheses: (100) = reference, ignore
    - Toleranced dimensions: "10.00 ±0.05" → extract 10.00

    MATERIAL HANDLING (ENHANCED):
    - **Priority Search Order**:
      1. Title block MATERIAL field
      2. Annotation symbols (①②③ etc.) and their legend tables
      3. Direct callouts with leader lines
      4. Notes section
      5. Material specifications in drawing border
    - If annotation symbol found (e.g., ③), YOU MUST find the corresponding table
    - Extract COMPLETE specification: grade, temper, treatment
    - Examples: "ALUMINUM 6061-T6", "STAINLESS STEEL 316L", "CARBON STEEL 1045"

    WEIGHT & SURFACE AREA:
    - For WEIGHT: Extract value and unit AS SHOWN (lb, kg, g, oz)
    - For SURFACE AREA: Extract value and unit AS SHOWN (in², cm², m², mm²)
    - Do NOT convert units
    - Do NOT assume units if not clearly labeled

    SPATIAL CLUES:
    - Section views (SECTION A-A) often show internal diameters
    - Top views typically show length × width
    - Front/side views show diameters
    - Detail views (DETAIL B, SCALE 2:1) may have specific dimensions

    NULL HANDLING:
    - If value not found or unclear → null
    - If unit not readable → null
    - Confidence < 80% → null
    - Better to return null than incorrect data

    ═══════════════════════════════════════════════════════════════════
    OUTPUT FORMAT (STRICT JSON, NO MARKDOWN):
    ═══════════════════════════════════════════════════════════════════

    {
      "title": "MOUNTING BRACKET - TYPE A",
      "drawing_number": "DRW-12345-R2",
      
      "length_value": 150.0,
      "length_unit": "mm",
      
      "inner_diameter_value": 12.7,
      "inner_diameter_unit": "mm",
      
      "outer_diameter_value": 25.4,
      "outer_diameter_unit": "mm",
      
      "material": "ALUMINUM 6061-T6",
      
      "weight_value": 0.85,
      "weight_unit": "lb",
      
      "surface_area_value": 23.5,
      "surface_area_unit": "in^2"
    }

    FIELD TYPES:
    - title: string or null
    - drawing_number: string or null
    - length_value: float or null
    - length_unit: string or null
    - inner_diameter_value: float or null
    - inner_diameter_unit: string or null
    - outer_diameter_value: float or null
    - outer_diameter_unit: string or null
    - material: string (FULL specification with grade/temper) or null
    - weight_value: float or null
    - weight_unit: string (AS SHOWN: lb, kg, g, oz) or null
    - surface_area_value: float or null
    - surface_area_unit: string (AS SHOWN: in^2, cm², m², mm²) or null

    ═══════════════════════════════════════════════════════════════════
    CRITICAL REMINDERS:
    ═══════════════════════════════════════════════════════════════════
    
    ✓ For MATERIAL: Check for annotation symbols (①②③) and find their legend/table
    ✓ Extract units exactly as shown on drawing
    ✓ Do NOT convert units
    ✓ Do NOT invent values - use null if uncertain
    ✓ Return ONLY valid JSON, no markdown code blocks
    """


# ==========================
# SIMPLIFIED NORMALIZATION
# ==========================
def cleanup_extracted_data(data: dict) -> dict:
    """
    Simplified cleanup - preserve units as extracted, just ensure structure.
    """
    keys_defaults = {
        "title": None,
        "drawing_number": None,
        "length_value": None,
        "length_unit": None,
        "inner_diameter_value": None,
        "inner_diameter_unit": None,
        "outer_diameter_value": None,
        "outer_diameter_unit": None,
        "material": None,
        "weight_value": None,
        "weight_unit": None,
        "surface_area_value": None,
        "surface_area_unit": None,
    }
    
    for k, v in keys_defaults.items():
        data.setdefault(k, v)

    # Remove any extra fields not in our schema
    return {k: data[k] for k in keys_defaults.keys() if k in data}


# ==========================
# API CALL WITH OPUS 4
# ==========================
def _call_model_on_image_bytes(image_bytes: bytes, mime_type: str) -> dict:
    """
    Call Claude Sonnet 4.5 for excellent accuracy and reliability.
    Using non-streaming API for faster response.
    """
    image_data = base64.standard_b64encode(image_bytes).decode("utf-8")
    
    try:
        response = client.messages.create(
            model=MODEL_NAME,
            max_tokens=8192,  # Sufficient for detailed analysis
            temperature=0.0,   # Zero temperature for consistency
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image",
                            "source": {
                                "type": "base64",
                                "media_type": mime_type,
                                "data": image_data,
                            },
                        },
                        {
                            "type": "text",
                            "text": get_dimension_prompt()
                        }
                    ],
                }
            ],
        )
        
        raw = response.content[0].text
        # Enhanced JSON extraction
        if "```json" in raw:
            raw = raw.split("```json", 1)[1].split("```", 1)[0]
        elif "```" in raw:
            raw = raw.split("```", 1)[1].split("```", 1)[0]

        try:
            data = json.loads(raw.strip())
        except json.JSONDecodeError as e:
            print(f"WARNING: JSON decode error: {e}")
            print(f"Raw response preview: {raw[:300]}...")
            data = {}

        if not isinstance(data, dict):
            data = {}

        return data
        
    except Exception as e:
        print(f"ERROR in API call: {e}")
        import traceback
        traceback.print_exc()
        return {}


# ==========================
# IMAGE ANALYSIS
# ==========================
def analyze_image(image_path: str) -> dict:
    """Analyze a single image with Claude Opus 4."""
    try:
        with open(image_path, "rb") as f:
            image_bytes = f.read()

        mime_type, _ = mimetypes.guess_type(image_path)
        if mime_type is None:
            mime_type = "image/png"

        data = _call_model_on_image_bytes(image_bytes, mime_type)
        data = cleanup_extracted_data(data)
        return data
        
    except Exception as e:
        print(f"ERROR analyzing {image_path}: {e}")
        return cleanup_extracted_data({})


# ==========================
# PDF → IMAGES (HIGH RESOLUTION)
# ==========================
def convert_file_to_images(file_path: str, dpi: int = 450):
    """
    Convert PDF to very high-resolution images for optimal OCR.
    Increased DPI to 450 for maximum text clarity and annotation recognition.
    """
    ext = Path(file_path).suffix.lower()
    if ext == ".pdf":
        try:
            print(f"  Converting PDF to images at {dpi} DPI...")
            pages = convert_from_path(file_path, dpi=dpi)
            if not pages:
                return [], True
            
            ts = int(time.time())
            imgs = []
            for i, p in enumerate(pages, start=1):
                img = f"temp_dim_{i}_{ts}.png"
                p.save(img, "PNG")
                imgs.append(img)
            print(f"  ✓ Converted to {len(imgs)} image(s)")
            return imgs, True
        except Exception as e:
            print(f"ERROR converting PDF: {e}")
            return [], True
    else:
        # For image files, just return the path
        return [file_path], False


# ==========================
# BUILD DATAFRAME - SIMPLIFIED
# ==========================
def build_output_dataframe(rows: List[dict]) -> pd.DataFrame:
    """Build dataframe with title, drawing number, and 6 parameters."""
    column_order = [
        "Page",
        "title",
        "drawing_number",
        "length_value",
        "length_unit",
        "inner_diameter_value",
        "inner_diameter_unit",
        "outer_diameter_value",
        "outer_diameter_unit",
        "material",
        "weight_value",
        "weight_unit",
        "surface_area_value",
        "surface_area_unit",
    ]
    
    df = pd.DataFrame(rows)
    
    # Ensure all columns exist
    for col in column_order:
        if col not in df.columns:
            df[col] = None
    
    return df[column_order]


# ==========================
# MAIN PROCESSOR
# ==========================
def process_file_enhanced(file_path: str, dpi: int = 450):
    """
    Process engineering drawings with Claude Opus 4.
    
    Args:
        file_path: Path to PDF or image file
        dpi: DPI for PDF conversion (default 450 for high accuracy)
    """
    path_obj = Path(file_path)
    if not path_obj.exists():
        print(f"ERROR: File not found: {file_path}")
        return

    print(f"\n{'='*70}")
    print(f"Processing: {path_obj.name}")
    print(f"Model: {MODEL_NAME} (Claude Sonnet 4.5)")
    print(f"{'='*70}")
    
    page_images, is_pdf = convert_file_to_images(file_path, dpi=dpi)
    if not page_images:
        print("ERROR: No images to process")
        return

    total_pages = len(page_images)
    print(f"\nTotal pages to analyze: {total_pages}\n")

    rows = []
    try:
        for i, img in enumerate(page_images, start=1):
            print(f"[Page {i}/{total_pages}] Analyzing with Sonnet 4.5...", end="", flush=True)
            start_time = time.time()
            
            result = analyze_image(img)
            result["Page"] = i
            rows.append(result)
            
            elapsed = time.time() - start_time
            print(f" ✓ ({elapsed:.1f}s)")
            
            # Show what was extracted
            extracted_info = []
            if result.get("title"):
                extracted_info.append(f"Title: {result['title']}")
            if result.get("drawing_number"):
                extracted_info.append(f"Dwg#: {result['drawing_number']}")
            if result.get("material"):
                extracted_info.append(f"Material: {result['material']}")
            if result.get("length_value"):
                extracted_info.append(f"Length: {result['length_value']} {result.get('length_unit', '')}")
            
            if extracted_info:
                print(f"             {' | '.join(extracted_info)}")

        df = build_output_dataframe(rows)

        # Generate output filename with timestamp
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        out_name = f"{path_obj.stem}_Extracted_{ts}.xlsx"
        
        # Save with formatting
        with pd.ExcelWriter(out_name, engine='openpyxl') as writer:
            df.to_excel(writer, sheet_name='Dimensions', index=False)
            
            # Auto-adjust column widths
            worksheet = writer.sheets['Dimensions']
            for column in worksheet.columns:
                max_length = 0
                column_letter = column[0].column_letter
                for cell in column:
                    try:
                        if len(str(cell.value)) > max_length:
                            max_length = len(str(cell.value))
                    except:
                        pass
                adjusted_width = min(max_length + 2, 50)
                worksheet.column_dimensions[column_letter].width = adjusted_width
        
        print(f"\n{'='*70}")
        print(f"✓ SUCCESS - Extraction Complete")
        print(f"{'='*70}")
        print(f"Output file: {out_name}")
        print(f"Total pages processed: {len(df)}")
        
        # Detailed summary statistics
        print(f"\n📊 Extraction Summary:")
        print(f"   Drawing titles found: {df['title'].notna().sum()}/{len(df)} pages")
        print(f"   Drawing numbers found: {df['drawing_number'].notna().sum()}/{len(df)} pages")
        print(f"   Length found:         {df['length_value'].notna().sum()}/{len(df)} pages")
        print(f"   Inner diameter found: {df['inner_diameter_value'].notna().sum()}/{len(df)} pages")
        print(f"   Outer diameter found: {df['outer_diameter_value'].notna().sum()}/{len(df)} pages")
        print(f"   Material identified:  {df['material'].notna().sum()}/{len(df)} pages")
        print(f"   Weight found:         {df['weight_value'].notna().sum()}/{len(df)} pages")
        print(f"   Surface area found:   {df['surface_area_value'].notna().sum()}/{len(df)} pages")
        
        # Show unique materials found
        unique_materials = df['material'].dropna().unique()
        if len(unique_materials) > 0:
            print(f"\n🔧 Materials Found:")
            for mat in unique_materials:
                print(f"   • {mat}")
        
        print(f"\n{'='*70}\n")

    except Exception as e:
        print(f"\n❌ ERROR during processing: {e}")
        import traceback
        traceback.print_exc()
        
    finally:
        # Cleanup temporary files
        if is_pdf:
            for img in page_images:
                try:
                    Path(img).unlink(missing_ok=True)
                except Exception:
                    pass


# ==========================
# BATCH PROCESSING
# ==========================
def process_directory(directory_path: str, pattern: str = "*.pdf"):
    """
    Process all matching files in a directory.
    
    Args:
        directory_path: Path to directory containing drawings
        pattern: File pattern to match (default: "*.pdf")
    """
    dir_path = Path(directory_path)
    if not dir_path.is_dir():
        print(f"ERROR: Not a valid directory: {directory_path}")
        return
    
    files = list(dir_path.glob(pattern))
    if not files:
        print(f"No files matching '{pattern}' found in {directory_path}")
        return
    
    print(f"\n{'#'*70}")
    print(f"BATCH PROCESSING: {len(files)} files found")
    print(f"{'#'*70}\n")
    
    for i, file_path in enumerate(files, start=1):
        print(f"\n{'▼'*70}")
        print(f"BATCH FILE {i}/{len(files)}")
        print(f"{'▼'*70}")
        process_file_enhanced(str(file_path))
        
        if i < len(files):
            print(f"\nWaiting 2 seconds before next file...\n")
            time.sleep(2)
    
    print(f"\n{'#'*70}")
    print(f"✓ BATCH COMPLETE - Processed {len(files)} files")
    print(f"{'#'*70}\n")


# ==========================
# ENTRY POINT
# ==========================
if __name__ == "__main__":
    print(f"\n{'='*70}")
    print(f"Engineering Drawing Dimension Extractor")
    print(f"{'='*70}")
    print(f"Model: {MODEL_NAME}")
    print(f"Capabilities:")
    print(f"  ✓ Material extraction from annotations (①②③) and legends")
    print(f"  ✓ Spatial understanding of drawing layouts")
    print(f"  ✓ High-resolution image processing (450 DPI)")
    print(f"  ✓ Preserves original units (no conversions)")
    print(f"  ✓ Fast and reliable with Sonnet 4.5")
    print(f"{'='*70}\n")
    
    # Example usage:
    # Single file:
    # process_file_enhanced("your_drawing.pdf")
    # process_file_enhanced("part_drawing.png")
    
    # Batch processing:
    # process_directory("./drawings", "*.pdf")
    
    # Ultra-high resolution (for very detailed drawings):
    # process_file_enhanced("complex_drawing.pdf", dpi=600)
    
    print("Module loaded successfully.\n")
    print("Usage Examples:")
    print("  process_file_enhanced('drawing.pdf')")
    print("  process_directory('./drawings', '*.pdf')")
    print("  process_file_enhanced('drawing.pdf', dpi=600)  # Ultra-high res\n")


Engineering Drawing Dimension Extractor
Model: claude-sonnet-4-5-20250929
Capabilities:
  ✓ Material extraction from annotations (①②③) and legends
  ✓ Spatial understanding of drawing layouts
  ✓ High-resolution image processing (450 DPI)
  ✓ Preserves original units (no conversions)
  ✓ Fast and reliable with Sonnet 4.5

Module loaded successfully.

Usage Examples:
  process_file_enhanced('drawing.pdf')
  process_directory('./drawings', '*.pdf')
  process_file_enhanced('drawing.pdf', dpi=600)  # Ultra-high res



In [3]:
file_path = "Test Batch.pdf"
#file_path = "https://www.shutterstock.com/image-vector/sketch-bushing-vector-eps10-260nw-111717389.jpg"
process_file_enhanced(file_path)


Processing: Test Batch.pdf
Model: claude-sonnet-4-5-20250929 (Claude Sonnet 4.5)
  Converting PDF to images at 450 DPI...
  ✓ Converted to 2 image(s)

Total pages to analyze: 2

[Page 1/2] Analyzing with Sonnet 4.5... ✓ (5.3s)
             Title: BUSHING BOOM PIVOT | Dwg#: 6811285 | Material: SR420 | Length: 1.733 in
[Page 2/2] Analyzing with Sonnet 4.5... ✓ (5.7s)
             Length: 51.0 mm

✓ SUCCESS - Extraction Complete
Output file: Test Batch_Extracted_20251209_105636.xlsx
Total pages processed: 2

📊 Extraction Summary:
   Drawing titles found: 1/2 pages
   Drawing numbers found: 1/2 pages
   Length found:         2/2 pages
   Inner diameter found: 2/2 pages
   Outer diameter found: 2/2 pages
   Material identified:  1/2 pages
   Weight found:         1/2 pages
   Surface area found:   1/2 pages

🔧 Materials Found:
   • SR420


